# Bot Question Tracking Spreadsheet
**Date:** 2026-03-07

Extract the "My Score" table from saved FutureEval Bot Tournament HTML page.

- **Input:** `../data/Spring 2026 FutureEval Bot Tournament 03-07-2026.html`
- **Output:** `../products/Bot_Question_Tracking_2026-03-07_v01.csv`

In [1]:
import re
import html
import csv
from pathlib import Path
import pandas as pd

HTML_FILE = Path(r"C:\Users\Donni\projects\metac_bot_Spring_2026\data\Spring 2026 FutureEval Bot Tournament 03-21-2026.html")
OUTPUT_CSV = Path(r"C:\Users\Donni\projects\metac_bot_Spring_2026\products\Bot_Question_Tracking_2026-03-21_v01.csv")

html_text = HTML_FILE.read_text(encoding="utf-8")
print(f"Loaded {HTML_FILE.name}: {len(html_text):,} chars")

Loaded Spring 2026 FutureEval Bot Tournament 03-21-2026.html: 9,126,630 chars


In [2]:
# Parse the "My Score" table rows
# Each row: <td><a href=".../questions/NNNNN/">Title</a></td> <th>Coverage</th> <td>Score</td> <th>Weight</th>
row_pattern = re.compile(
    r'href="https://www\.metaculus\.com/questions/(\d+)/">'
    r'(.*?)</a></td>'
    r'<th[^>]*>([^<]*)</th>'
    r'<td[^>]*>([^<]*)</td>'
    r'<th[^>]*>([^<]*)</th>'
)

# Only search in the My Score table area (after "mb-3 w-full")
table_start = html_text.find('class="mb-3 w-full"')
table_end = html_text.find('</table>', table_start)
table_html = html_text[table_start:table_end]

questions = []
for m in row_pattern.finditer(table_html):
    questions.append({
        "question_number": int(m.group(1)),
        "title": html.unescape(m.group(2).strip()),
        "coverage": m.group(3).strip(),
        "score": m.group(4).strip(),
        "question_weight": m.group(5).strip(),
    })

scored = [q for q in questions if q["score"] != "-"]
print(f"Parsed {len(questions)} questions ({len(scored)} scored, {len(questions) - len(scored)} unscored)")
if scored:
    scores = [float(q["score"]) for q in scored]
    print(f"Score range: {min(scores):.3f} to {max(scores):.3f}, total: {sum(scores):.3f}")

Parsed 0 questions (0 scored, 0 unscored)


In [3]:
# Sort: scored first (descending), then unscored
sorted_qs = sorted(questions, key=lambda q: (q["score"] == "-", -float(q["score"]) if q["score"] != "-" else 0))

# Display as DataFrame
df = pd.DataFrame(sorted_qs)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 80)
df

""


In [4]:
# Write CSV
fieldnames = ["question_number", "title", "coverage", "score", "question_weight"]

with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(sorted_qs)

size_kb = OUTPUT_CSV.stat().st_size / 1024
print(f"Wrote {len(sorted_qs)} rows to {OUTPUT_CSV.name} ({size_kb:.1f} KB)")

Wrote 0 rows to Bot_Question_Tracking_2026-03-21_v01.csv (0.1 KB)
